# Exercise 0: Environment and libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings('ignore')

# Exercise 1: Imputer 1

In [2]:
train_data = [[7, 6, 5],
              [4, np.nan, 5],
              [1, 20, 8]]

imputer = SimpleImputer(strategy='mean')
imputer.fit(train_data)
print("Statistics_:", imputer.statistics_)
print("np.nanmean:", np.nanmean(train_data, axis=0))

train_filled = imputer.transform(train_data)
print("\nTrain filled:\n", train_filled)

test_data = [[np.nan, 1, 2],
             [7, np.nan, 9],
             [np.nan, 2, 4]]
test_filled = imputer.transform(test_data)
print("\nTest filled:\n", test_filled)

Statistics_: [ 4. 13.  6.]
np.nanmean: [ 4. 13.  6.]

Train filled:
 [[ 7.  6.  5.]
 [ 4. 13.  5.]
 [ 1. 20.  8.]]

Test filled:
 [[ 4.  1.  2.]
 [ 7. 13.  9.]
 [ 4.  2.  4.]]


# Exercise 2: Scaler

In [3]:
X_train = np.array([[ 1., -1.,  2.],
                     [ 2.,  0.,  0.],
                     [ 0.,  1., -1.]])
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
print("X_train scaled:\n", X_train_scaled)
print("Mean:", X_train_scaled.mean(axis=0))
print("Std:", X_train_scaled.std(axis=0))

X_test = np.array([[ 2., -1.,  1.],
                     [ 3.,  3.,  -1.],
                     [ 1.,  1., 1.]])
X_test_scaled = scaler.transform(X_test)
print("\nX_test scaled:\n", X_test_scaled)

X_train scaled:
 [[ 0.         -1.22474487  1.33630621]
 [ 1.22474487  0.         -0.26726124]
 [-1.22474487  1.22474487 -1.06904497]]
Mean: [0. 0. 0.]
Std: [1. 1. 1.]

X_test scaled:
 [[ 1.22474487 -1.22474487  0.53452248]
 [ 2.44948974  3.67423461 -1.06904497]
 [ 0.          1.22474487  0.53452248]]


# Exercise 3: One hot Encoder

In [4]:
X_train = [['Python'], ['Java'], ['Java'], ['C++']]
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train)
X_train_trans = ohe.transform(X_train)
print("Train Transformed DF:")
print(pd.DataFrame(X_train_trans, columns=ohe.categories_[0]))

X_test = [['Python'], ['Java'], ['C'], ['C++']]
X_test_trans = ohe.transform(X_test)
print("\nTest Transformed DF:")
print(pd.DataFrame(X_test_trans, columns=ohe.categories_[0]))

Train Transformed DF:
   C++  Java  Python
0  0.0   0.0     1.0
1  0.0   1.0     0.0
2  0.0   1.0     0.0
3  1.0   0.0     0.0

Test Transformed DF:
   C++  Java  Python
0  0.0   0.0     1.0
1  0.0   1.0     0.0
2  0.0   0.0     0.0
3  1.0   0.0     0.0


# Exercise 4: Ordinal Encoder

In [5]:
X_train = [['good'], ['bad'], ['neutral']]
# Categories order: bad, neutral, good
cats = [['bad', 'neutral', 'good']]
oe = OrdinalEncoder(categories=cats)
oe.fit(X_train)
X_train_trans = oe.transform(X_train)
print("Categories_:", oe.categories_)
print("Transformed train:\n", X_train_trans)

X_test = [['good'], ['good'], ['bad']]
X_test_trans = oe.transform(X_test)
print("Transformed test:\n", X_test_trans)

Categories_: [array(['bad', 'neutral', 'good'], dtype=object)]
Transformed train:
 [[2.]
 [0.]
 [1.]]
Transformed test:
 [[2.]
 [2.]
 [0.]]


# Exercise 5: Categorical variables

In [6]:
# Column names based on inspection of breast-cancer.csv which has no header
cols = ['age', 'menopause', 'tumor-size', 'inv-nodes', 'node-caps', 'deg-malig', 'breast', 'breast-quad', 'irradiat', 'Class']
df = pd.read_csv('breast-cancer.csv', header=None, names=cols)

# Drop Class
X = df.drop('Class', axis=1)
y = df['Class']

# Drop NaN
X = X.dropna()
print("Shape after dropna:", X.shape)

# Split
X_train, X_test = train_test_split(X, test_size=0.2, random_state=43)

# Count unique
print("Unique values per feature:\n", X_train.nunique())

# One Hot Encoding
ohe_cols = ['node-caps' , 'breast', 'breast-quad', 'irradiat']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[ohe_cols])
print("\nOHE Transformed Test (first 10):\n", ohe.transform(X_test[ohe_cols])[:10])
try:
    print("Feature names:", ohe.get_feature_names_out(ohe_cols))
except:
    pass

# Ordinal Encoding
# Using the reversed order lists for OrdinalEncoder.
cats_menopause = ['lt40', 'premeno', 'ge40']
cats_age = ['10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99']
cats_tumor = ['0-4', '5-9', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59']
cats_inv = ['0-2', '3-5', '6-8', '9-11', '12-14', '15-17', '18-20', '21-23', '24-26', '27-29', '30-32', '33-35', '36-39']
cats_malig = [1, 2, 3]

ordinal_cols = ["menopause", "age", "tumor-size", "inv-nodes", "deg-malig"]
ordinal_cats = [cats_menopause, cats_age, cats_tumor, cats_inv, cats_malig]

oe = OrdinalEncoder(categories=ordinal_cats)
oe.fit(X_train[ordinal_cols])

print("\nOrdinal Transformed Test (first 5):\n", oe.transform(X_test[ordinal_cols])[:5])

# Column Transformer
# Need to make sure we use a list of column names for make_column_transformer when passing pandas
ct = ColumnTransformer([
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ohe_cols),
    ('oe', OrdinalEncoder(categories=ordinal_cats), ordinal_cols)
])
ct.fit(X_train)
X_test_trans_ct = ct.transform(X_test)
print("\nColumnTransformer Transformed Test Shape:", X_test_trans_ct.shape)

Shape after dropna: (277, 9)
Unique values per feature:
 age             6
menopause       3
tumor-size     11
inv-nodes       6
node-caps       2
deg-malig       3
breast          2
breast-quad     5
irradiat        2
dtype: int64

OHE Transformed Test (first 10):
 [[1. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0.]
 [0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [1. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0.]
 [1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0.]
 [1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1.]
 [1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.]]
Feature names: ['node-caps_no' 'node-caps_yes' 'breast_left' 'breast_right'
 'breast-quad_central' 'breast-quad_left_low' 'breast-quad_left_up'
 'breast-quad_right_low' 'breast-quad_right_up' 'irradiat_no'
 'irradiat_yes']

Ordinal Transformed Test (first 5):
 [[2. 5. 2. 0. 1.]
 [2. 5. 2. 0. 0.]
 [2. 5. 4. 5. 2.]
 [1. 4. 5. 1. 1.]
 [2. 5. 5. 0. 2.]]

ColumnTransformer T

# Exercise 6: Pipeline

In [7]:
iris = load_iris()
X, y = iris['data'], iris['target']
X[[1,20,50,100,135], 0] = np.nan
X[[2,5,88,135], 1] = np.nan
X[[4,15], 2] = np.nan
X[[40,135], 3] = np.nan

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=43)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression())
])

pipe.fit(X_train, y_train)
print("Pipeline Test Score:", pipe.score(X_test, y_test))

Pipeline Test Score: 0.98
